In [ ]:
#!/usr/bin/env python
# =========================================================
# EFC-VAE training script — Red Sea SST holdout emulation
#
# Trains EFC-VAE on the Red Sea SST data with a held-out subset of sites
# and evaluates out-of-sample emulation (Section 5.1 of the paper).
# Uses the GEV-PIT marginal transform (site-wise fitted GEV parameters,
# Section 3.1) rather than the rank-PIT used in the simulation study.
# Shares the model definition and training stages with the simulation
# script (see train_simulation.py); Stage descriptions below mirror
# Supplementary Section "Loss function and training settings" and the
# Algorithm in Section 3.4/Appendix.
# =========================================================

import os
import time
import copy
import argparse
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.stats import genextreme, norm
from scipy.spatial.distance import pdist, squareform

# =========================================================
# CLI / paths
# =========================================================
parser = argparse.ArgumentParser(description="EFC-VAE training on Red Sea SST holdout data")
parser.add_argument("--data_dir", type=str, default="./data/redsea",
                     help="Directory containing W_basis.csv, stations.csv, "
                          "mon_max_allsites.csv, fitted_gev_par.csv")
parser.add_argument("--out_dir", type=str, default="./runs",
                     help="Directory where results are written")
parser.add_argument("--gpu", type=int, default=0, help="CUDA device index; -1 for CPU")
parser.add_argument("--epochs_s1", type=int, default=2500)
parser.add_argument("--epochs_s2", type=int, default=2500)
parser.add_argument("--epochs_s3a", type=int, default=2500)
parser.add_argument("--epochs_s3b", type=int, default=2500)
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--n_emul", type=int, default=100)
parser.add_argument("--n_holdout", type=int, default=100)
parser.add_argument("--holdout_seed", type=int, default=4040)
parser.add_argument("--seed", type=int, default=2026)
args = parser.parse_args()

DATA_DIR = args.data_dir
DATE_TAG = datetime.now().strftime("%y%m%d")
RESULT_DIR = os.path.join(args.out_dir, f"redsea_holdout_{DATE_TAG}")
os.makedirs(RESULT_DIR, exist_ok=True)

HIDDEN_DIM   = 256
BATCH_SIZE   = args.batch_size
N_EPOCHS_S1  = args.epochs_s1
N_EPOCHS_S2  = args.epochs_s2
N_EPOCHS_S3A = args.epochs_s3a
N_EPOCHS_S3B = args.epochs_s3b
LR_BASE      = 1e-3

ALPHA_MAX = 1.0
GATE_INIT = 0.3
LAM_DEPENDENCE     = 70.0
LAM_REGULARIZATION = 1e-3
LAM_GATE           = 5e-3
CHI_N_PAIRS = 2000
N_EMUL = args.n_emul
GLOBAL_SEED = args.seed


def setup_device(gpu_id):
    if gpu_id is None or gpu_id < 0 or not torch.cuda.is_available():
        return torch.device("cpu")
    if gpu_id < torch.cuda.device_count():
        torch.cuda.set_device(gpu_id)
        return torch.device(f"cuda:{gpu_id}")
    print(f"WARNING: GPU {gpu_id} not found, using cuda:0")
    return torch.device("cuda:0")


DEVICE = setup_device(args.gpu)
print(f"device: {DEVICE}")
print(f"data_dir: {DATA_DIR}")
print(f"out_dir : {RESULT_DIR}")

# =========================================================
# Derived loss-weight constants (see train_simulation.py for the same
# schedule and Supplementary "Loss function and training settings")
# =========================================================
_LR_S1 = LR_BASE * 0.5
_LR_S2 = LR_BASE * 0.1
_LR_S3 = LR_BASE * 2.0
_LR_FACTOR_NU    = LR_BASE * 10
_LR_FACTOR_THETA = LR_BASE * 8
_LR_FACTOR_APS   = LR_BASE * 5
_LR_FACTOR_GATE  = LR_BASE * 10
_LR_DECAY_FACTOR = 0.5

_LAM_CHI = LAM_DEPENDENCE * 0.7
_LAM_Q99 = LAM_DEPENDENCE * 0.3
_LAM_QM  = LAM_DEPENDENCE * 0.03
_LAM_REC = 1.0
_LAM_TW  = 1.0

_LAM_TW_S2   = 3.0
_LAM_QM_S2   = 5.0
_LAM_Q99_S2  = 25.0
_LAM_Q995_S2 = 5.0   # reduced relative to the simulation setting (n_t=372)
_ALPHA_TW_S2 = 2.5

_LAM_ALPHA_REG    = LAM_REGULARIZATION
_LAM_NU_REG       = LAM_REGULARIZATION * 1e-4
_LAM_NU_SMOOTH    = LAM_REGULARIZATION * 0.1
_LAM_THETA_SMOOTH = LAM_REGULARIZATION
_LAM_CHI_POSTERIOR = 0.5   # extra chi-loss term evaluated on reconstructions
_LAM_GATE_TARGET = LAM_GATE
_LAM_GATE_SMOOTH = LAM_GATE * 0.002

_ALPHA_TW = 1.5
_TAU_TW   = 0.95
_BETA_KL  = 0.1

_NU_INIT, _NU_MIN, _NU_MAX = 5.0, 2.5, 500.0
_ALPHA_INIT = 0.50
_ALPHA_PS_INIT, _ALPHA_PS_MIN, _ALPHA_PS_MAX = 0.5, 0.1, 0.95
_THETA_INIT, _THETA_MIN, _THETA_MAX = 2.0, 0.05, 20.0
_M0_CLIP_MAX = 6.0
_V_0_CLIP = 5.0
_GATE_THRESHOLD = 0.03

_M0_SITE_CHUNK = 4000   # chunk size for the shared positive-stable factor
                        # (needed at Red Sea scale, ~16,700 sites)

# Weighted quantile/chi-threshold grids (Section "Loss function and
# training settings"; {0.95, 0.99} receive 70% of the quantile-loss weight)
Q_LOSS_LEVELS = (0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)
_QM_QWS = ((0.10, 0.42), (0.25, 0.42), (0.50, 0.42), (0.75, 0.42),
           (0.90, 0.42), (0.95, 1.96), (0.99, 2.94))
_Q_LOSS_WEIGHTS = {0.10: 0.42, 0.25: 0.42, 0.50: 0.42, 0.75: 0.42,
                   0.90: 0.42, 0.95: 1.96, 0.99: 2.94}
_CHI_U_LEVELS = (0.50, 0.70, 0.80, 0.90, 0.94, 0.97)
_CHI_TEMP = 10.0
_CHI_WEIGHT_EXP = 1.0

_EVAL_EVERY = 10
_BEST_TRACK_EP_MIN = 10
_PRINT_EVERY = 10
Z_HAT_CAP = 6.0

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# =========================================================
# GEV-PIT marginal transform (Section 3.1, Eq. method-gevpit)
# =========================================================
def gev_pit_to_normal_with_par(X, mu_arr, sigma_arr, xi_arr):
    """Z_jt = Phi^{-1}( G_j(X_jt) ), with G_j the fitted GEV margin at site j."""
    n_site, n_t = X.shape
    Z = np.zeros_like(X, dtype=np.float64)
    for i in range(n_site):
        sigma_i = max(float(sigma_arr[i]), 1e-6)
        try:
            U = genextreme.cdf(X[i], c=-xi_arr[i], loc=mu_arr[i], scale=sigma_i)
        except Exception:
            U = (np.argsort(np.argsort(X[i])) + 1) / (n_t + 1)
        Z[i] = norm.ppf(np.clip(U, 1e-6, 1 - 1e-6))
    return np.clip(Z, -6, 6).astype(np.float32)


def inverse_pit_gev(Z, mu_arr, sigma_arr, xi_arr):
    """X_jt = G_j^{-1}( Phi(Z_jt) ), the inverse GEV-PIT transform
    (Eq. kmrt-main)."""
    U = np.clip(norm.cdf(np.clip(Z, -6, 6)), 1e-6, 1 - 1e-6)
    n_site, n_t = U.shape
    X = np.zeros_like(U, dtype=np.float64)
    for i in range(n_site):
        sigma_i = max(float(sigma_arr[i]), 1e-6)
        try:
            X[i] = genextreme.ppf(U[i], c=-xi_arr[i], loc=mu_arr[i], scale=sigma_i)
        except Exception:
            X[i] = mu_arr[i] + sigma_i * U[i]
    for i in range(n_site):
        bad = ~np.isfinite(X[i])
        if bad.any():
            fill = float(np.nanmax(X[i][~bad])) if (~bad).any() else mu_arr[i]
            X[i][bad] = fill
    return X.astype(np.float32)


# =========================================================
# Loss terms — identical to the simulation study
# (Eq. main-total-loss; see train_simulation.py for full comments)
# =========================================================
def tail_weighted_mse(z_pred, z_true, alpha=_ALPHA_TW, tau=_TAU_TW):
    z_tau = torch.quantile(z_true, tau)
    w = torch.exp(alpha * F.relu(z_true - z_tau))
    return torch.mean(w * (z_pred - z_true) ** 2)


def quantile_match_loss(z_pred, z_true, qws=_QM_QWS):
    return sum(w * F.mse_loss(torch.quantile(z_pred, q, dim=0),
                               torch.quantile(z_true, q, dim=0))
               for q, w in qws)


def kl_gaussian_loss(mu, lv):
    return -0.5 * torch.mean(1 + lv - mu ** 2 - lv.exp())


def stage1_loss(z_pred, z_true, mu, lv):
    l_rec = F.mse_loss(z_pred, z_true)
    l_tw = tail_weighted_mse(z_pred, z_true, _ALPHA_TW, _TAU_TW)
    l_qm = quantile_match_loss(z_pred, z_true)
    l_kl = kl_gaussian_loss(mu, lv)
    total = l_rec + _LAM_TW * l_tw + 0.5 * l_qm + _BETA_KL * l_kl
    return total, dict(rec=l_rec.item(), tw=l_tw.item(), qm=l_qm.item(), kl=l_kl.item())


def stage2_loss(z_pred, z_true):
    l_rec = F.mse_loss(z_pred, z_true)
    l_tw = tail_weighted_mse(z_pred, z_true, _ALPHA_TW_S2, _TAU_TW)
    l_qm = quantile_match_loss(z_pred, z_true)
    l_q99 = F.mse_loss(torch.quantile(z_pred, 0.99, dim=0), torch.quantile(z_true, 0.99, dim=0))
    l_q995 = F.mse_loss(torch.quantile(z_pred, 0.995, dim=0), torch.quantile(z_true, 0.995, dim=0))
    total = (l_rec + _LAM_TW_S2 * l_tw + _LAM_QM_S2 * l_qm
             + _LAM_Q99_S2 * l_q99 + _LAM_Q995_S2 * l_q995)
    return total, dict(rec=l_rec.item(), q99=l_q99.item())


_CHI_THRESHOLDS = torch.tensor([float(norm.ppf(u)) for u in _CHI_U_LEVELS],
                                dtype=torch.float32)


def compute_chi_soft(z, pair_i, pair_j, thresholds=_CHI_THRESHOLDS, temp=_CHI_TEMP):
    z_i, z_j = z[:, pair_i], z[:, pair_j]
    th = thresholds.to(z.device)
    chis = []
    for ui in range(th.shape[0]):
        a = torch.sigmoid((z_i - th[ui]) * temp)
        b = torch.sigmoid((z_j - th[ui]) * temp)
        chi = (a * b).mean(dim=0) / b.mean(dim=0).clamp(min=1e-6)
        chis.append(chi)
    return torch.stack(chis, dim=0)


def estimate_gate_target(chi_target):
    chi_high = chi_target[-1].mean().item()
    return torch.tensor(max(0.15, min(0.65, chi_high * 1.6)), dtype=torch.float32)


def stage3_loss(z_recon, z_input, z_gen, chi_target, alpha_basis,
                 log_nu_basis, log_theta_basis, gate_basis, gate_per_site,
                 gate_target, pair_i, pair_j):
    l_rec = F.mse_loss(z_recon, z_input)
    l_qm = quantile_match_loss(z_recon, z_input)

    chi_gen = compute_chi_soft(z_gen, pair_i, pair_j)
    chi_weights = (chi_target.detach() ** _CHI_WEIGHT_EXP + 0.05)
    chi_weights = chi_weights / chi_weights.mean()
    l_chi_prior = (chi_weights * (chi_gen - chi_target) ** 2).mean()

    l_chi_post = torch.tensor(0.0, device=z_recon.device)
    if z_recon.shape[0] >= 16:
        chi_recon = compute_chi_soft(z_recon, pair_i, pair_j)
        l_chi_post = (chi_weights * (chi_recon - chi_target) ** 2).mean()
    l_chi = l_chi_prior + _LAM_CHI_POSTERIOR * l_chi_post

    l_q99, w_sum = 0.0, 0.0
    for q in Q_LOSS_LEVELS:
        w = _Q_LOSS_WEIGHTS.get(q, 1.0)
        qp = torch.quantile(z_gen, q, dim=0)
        qt = torch.tensor(float(norm.ppf(q)), device=z_gen.device)
        l_q99 = l_q99 + w * ((qp - qt) ** 2).mean()
        w_sum += w
    l_q99 = l_q99 / max(w_sum, 1e-8)

    l_alpha_reg = (alpha_basis ** 2).mean()
    nu_vals = _NU_MIN + torch.exp(log_nu_basis)
    l_nu_smooth = nu_vals.var()
    l_nu_reg = (1.0 / (nu_vals + 1.0)).mean()
    theta_vals = _THETA_MIN + torch.exp(log_theta_basis)
    l_theta_smooth = theta_vals.var()
    l_gate_target = ((gate_per_site - gate_target) ** 2).mean()
    l_gate_smooth = gate_basis.var()

    total = (_LAM_REC * l_rec + _LAM_CHI * l_chi + _LAM_QM * l_qm
             + _LAM_Q99 * l_q99 + _LAM_ALPHA_REG * l_alpha_reg
             + _LAM_NU_REG * l_nu_reg + _LAM_NU_SMOOTH * l_nu_smooth
             + _LAM_THETA_SMOOTH * l_theta_smooth
             + _LAM_GATE_TARGET * l_gate_target + _LAM_GATE_SMOOTH * l_gate_smooth)

    q_log = l_q99.item() if torch.is_tensor(l_q99) else float(l_q99)
    return total, dict(rec=l_rec.item(), chi=l_chi.item(),
                        chi_prior=l_chi_prior.item(), chi_post=float(l_chi_post.item()),
                        q99=q_log, gate_target=l_gate_target.item())


# =========================================================
# Model (identical to train_simulation.py; see that file for full
# per-line comments tied to the paper equations)
# =========================================================
class CopulaVAE(nn.Module):
    def __init__(self, n_site, n_basis, n_rep, hidden=HIDDEN_DIM, factor=False):
        super().__init__()
        self.n_site, self.n_basis, self.n_rep = n_site, n_basis, n_rep
        self.factor = factor
        out_dim = n_basis + (1 if factor else 0)
        self.latent_dim = out_dim

        self.enc = nn.Sequential(
            nn.Linear(n_site, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden, 128), nn.LayerNorm(128), nn.SiLU())
        self.fc_mu = nn.Linear(128, out_dim)
        self.fc_lv = nn.Linear(128, out_dim)
        self.b = nn.Parameter(torch.zeros(n_site, n_rep))  # site--block residual R_jt

        if factor:
            self.log_alpha_basis = nn.Parameter(
                torch.log(torch.full((n_basis,), float(_ALPHA_INIT))))
            self.log_nu_basis = nn.Parameter(
                torch.full((n_basis,), float(np.log(_NU_INIT - _NU_MIN + 1e-6))))
            self.logit_alpha_ps = nn.Parameter(
                torch.tensor(float(np.log(
                    (_ALPHA_PS_INIT - _ALPHA_PS_MIN) / (_ALPHA_PS_MAX - _ALPHA_PS_INIT)))))
            self.log_theta_basis = nn.Parameter(
                torch.full((n_basis,), float(np.log(_THETA_INIT - _THETA_MIN + 1e-6))))
            init_logit_g = float(np.log(GATE_INIT / (1 - GATE_INIT)))
            self.logit_gate_basis = nn.Parameter(
                torch.full((n_basis,), init_logit_g) + torch.randn(n_basis) * 0.2)

    def alpha_basis(self):
        return torch.clamp(torch.exp(self.log_alpha_basis), 1e-4, ALPHA_MAX)

    def alpha_per_site(self, W):
        return W @ self.alpha_basis()

    def nu_basis(self):
        return torch.clamp(_NU_MIN + torch.exp(self.log_nu_basis), _NU_MIN, _NU_MAX)

    def nu_per_site(self, W):
        return W @ self.nu_basis()

    def alpha_ps(self):
        raw = torch.sigmoid(self.logit_alpha_ps)
        return _ALPHA_PS_MIN + (_ALPHA_PS_MAX - _ALPHA_PS_MIN) * raw

    def theta_basis(self):
        return torch.clamp(_THETA_MIN + torch.exp(self.log_theta_basis), _THETA_MIN, _THETA_MAX)

    def theta_per_site(self, W):
        return W @ self.theta_basis()

    def gate_basis_val(self):
        return torch.sigmoid(self.logit_gate_basis)

    def gate_per_site(self, W):
        return W @ self.gate_basis_val()

    def encode(self, x):
        h = self.enc(x)
        mu = self.fc_mu(h)
        lv = torch.clamp(self.fc_lv(h), -4, 4)
        return mu, lv

    def reparam(self, mu, lv, det=False):
        return mu if det else mu + torch.randn_like(mu) * torch.exp(0.5 * lv)

    def sample_V_0(self, h_gauss, W):
        """Student-t factor (Eq. method-tfactor)."""
        nu_site = self.nu_per_site(W)
        BS, n_site = h_gauss.shape[0], W.shape[0]
        nu_e = nu_site.unsqueeze(0).expand(BS, -1)
        chi2 = torch.distributions.Gamma(
            concentration=nu_e / 2.0,
            rate=torch.tensor(0.5, device=h_gauss.device)).rsample()
        chi2 = torch.clamp(chi2, min=1e-4)
        h_e = h_gauss.unsqueeze(1).expand(-1, n_site)
        v_t = h_e / torch.sqrt(chi2 / nu_e)
        return torch.clamp(v_t, -_V_0_CLIP * 1.5, _V_0_CLIP * 1.5)

    def sample_M_0_shared(self, BS, theta_per_site_val, W):
        """Positive-stable factor, generated at the knot level and
        aggregated with the Wendland weights, chunked over sites for
        the ~16,700-site Red Sea grid (Supplementary 'Knot-level
        generation and spatial aggregation of the positive-stable
        factor')."""
        alpha = self.alpha_ps()
        device = W.device
        n_site, n_knot = W.shape

        u = (torch.rand(BS, n_knot, device=device) * np.pi).clamp(1e-4, np.pi - 1e-4)
        e = -torch.log(torch.rand(BS, n_knot, device=device).clamp(1e-6))
        log_z = (torch.log(torch.sin(alpha * u).clamp(min=1e-8))
                  + ((1 - alpha) / alpha) * (
                      torch.log(torch.sin((1 - alpha) * u).clamp(min=1e-8))
                      - torch.log(torch.sin(u).clamp(min=1e-8))))
        log_lambda = torch.clamp(
            log_z - ((1 - alpha) / alpha) * torch.log(e.clamp(min=1e-6)), min=-10, max=8)

        logW_pow = torch.log(W.clamp(min=1e-12)) / alpha
        theta = theta_per_site_val
        chunks = []
        for s0 in range(0, n_site, _M0_SITE_CHUNK):
            s1 = min(s0 + _M0_SITE_CHUNK, n_site)
            comb = log_lambda.unsqueeze(1) + logW_pow[s0:s1].unsqueeze(0)
            log_S = torch.clamp(alpha * torch.logsumexp(comb, dim=2), min=-10, max=8)
            S_pow_alpha = torch.exp(alpha * log_S)
            log_M = torch.clamp(log_S - theta[s0:s1].unsqueeze(0) * S_pow_alpha,
                                 min=-8, max=_M0_CLIP_MAX)
            M = torch.exp(log_M)
            M_mean = M.mean(dim=0, keepdim=True) + 1e-6
            chunks.append(torch.clamp(torch.log(M / M_mean + 0.1) * 0.5, -3.0, 3.0))
        return torch.cat(chunks, dim=1)

    def decode(self, z, W, t_idx=None):
        if self.factor:
            c = z[:, :self.n_basis]
            h_gauss = z[:, self.n_basis]
            out = c @ W.T
            V_0 = self.sample_V_0(h_gauss, W)
            g = self.gate_per_site(W)
            if g.max().item() > _GATE_THRESHOLD:
                M_0 = self.sample_M_0_shared(z.shape[0], self.theta_per_site(W), W)
            else:
                M_0 = torch.zeros_like(V_0)
            F_combined = (1 - g).unsqueeze(0) * V_0 + g.unsqueeze(0) * M_0
            alpha = self.alpha_per_site(W)
            out = out + alpha.unsqueeze(0) * F_combined
        else:
            out = z @ W.T
        if t_idx is not None and self.b.shape[0] == out.shape[1]:
            out = out + self.b[:, t_idx].T
        return out

    def forward(self, x, W, t_idx=None, det=False):
        mu, lv = self.encode(x)
        z = self.reparam(mu, lv, det=det)
        return self.decode(z, W, t_idx=t_idx), mu, lv, z


# =========================================================
# Holdout residual transfer and evaluation metrics
# =========================================================
def idw_interpolate_matrix(M_ref, stations_pred, stations_ref, n_neighbors=8, power=2.0, eps=1e-8):
    """Coordinate-based IDW transfer of the residual field to holdout
    locations (Eq. kmrt-idw)."""
    n_pred, n_ref = stations_pred.shape[0], stations_ref.shape[0]
    k = min(n_neighbors, n_ref)
    D = np.sqrt(((stations_pred[:, None, :] - stations_ref[None, :, :]) ** 2).sum(axis=2))
    nn_idx = np.argpartition(D, k - 1, axis=1)[:, :k]
    nn_dist = np.take_along_axis(D, nn_idx, axis=1)
    w = 1.0 / np.maximum(nn_dist, eps) ** power
    w = w / np.maximum(w.sum(axis=1, keepdims=True), eps)
    M_out = np.zeros((n_pred, M_ref.shape[1]), dtype=np.float32)
    for i in range(n_pred):
        M_out[i] = (w[i, :, None] * M_ref[nn_idx[i]]).sum(axis=0)
    return M_out


def quantile_metrics(X_truth, X_pred):
    """Bias, variance, RMSE, correlation at Q95/Q99, and field RMSE/MAE
    (Table tab:redsea-main)."""
    q95_t = np.nanquantile(X_truth, 0.95, axis=1)
    q99_t = np.nanquantile(X_truth, 0.99, axis=1)
    q95_p = np.nanquantile(X_pred, 0.95, axis=1)
    q99_p = np.nanquantile(X_pred, 0.99, axis=1)

    def _corr(a, b):
        m = np.isfinite(a) & np.isfinite(b)
        return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() >= 3 else np.nan

    err_q95, err_q99 = q95_p - q95_t, q99_p - q99_t
    return dict(
        bias_q95=float(np.nanmean(err_q95)), var_q95=float(np.nanvar(err_q95)),
        rmse_q95=float(np.sqrt(np.nanmean(err_q95 ** 2))),
        mae_q95=float(np.nanmean(np.abs(err_q95))), corr_q95=_corr(q95_t, q95_p),
        bias_q99=float(np.nanmean(err_q99)), var_q99=float(np.nanvar(err_q99)),
        rmse_q99=float(np.sqrt(np.nanmean(err_q99 ** 2))),
        mae_q99=float(np.nanmean(np.abs(err_q99))), corr_q99=_corr(q99_t, q99_p),
        field_rmse=float(np.sqrt(np.nanmean((X_truth - X_pred) ** 2))),
        field_mae=float(np.nanmean(np.abs(X_truth - X_pred))),
    )


# =========================================================
# Main
# =========================================================
def main():
    log_lines = ["EFC-VAE — Red Sea SST holdout emulation"]

    def L(s):
        log_lines.append(s)
        print(s)

    L(f"device: {DEVICE}")
    L(f"output dir: {RESULT_DIR}")
    L(f"S1={N_EPOCHS_S1}, S2={N_EPOCHS_S2}, S3a={N_EPOCHS_S3A}, S3b={N_EPOCHS_S3B}, "
      f"batch={BATCH_SIZE}, n_emul={N_EMUL}")
    L(f"n_holdout={args.n_holdout}, holdout_seed={args.holdout_seed}")

    L("\n=== Loading Red Sea data ===")
    W_all = pd.read_csv(os.path.join(DATA_DIR, "W_basis.csv")).values.astype(np.float32)
    stations_all = pd.read_csv(os.path.join(DATA_DIR, "stations.csv")).values.astype(np.float32)
    X_all = pd.read_csv(os.path.join(DATA_DIR, "mon_max_allsites.csv")).values.astype(np.float32)
    gev_par = pd.read_csv(os.path.join(DATA_DIR, "fitted_gev_par.csv"))
    mu_all = gev_par["location"].values.astype(np.float64)
    sigma_all = gev_par["scale"].values.astype(np.float64)
    xi_all = gev_par["shape"].values.astype(np.float64)
    if stations_all.shape[1] >= 3:
        stations_all = stations_all[:, -2:]

    n_all, n_t = X_all.shape
    K = W_all.shape[1]
    if not (0 < args.n_holdout < n_all):
        raise ValueError(f"n_holdout must be between 1 and n_all-1; got {args.n_holdout}")
    W_all = W_all / np.maximum(W_all.sum(axis=1, keepdims=True), 1e-8)

    rng_split = np.random.default_rng(args.holdout_seed)
    holdout_idx = np.sort(rng_split.choice(n_all, size=args.n_holdout, replace=False))
    train_idx = np.setdiff1d(np.arange(n_all), holdout_idx)

    X_tr, X_va = X_all[train_idx], X_all[holdout_idx]
    W_tr, W_va = W_all[train_idx], W_all[holdout_idx]
    sta_tr, sta_va = stations_all[train_idx], stations_all[holdout_idx]
    mu_tr, sigma_tr, xi_tr = mu_all[train_idx], sigma_all[train_idx], xi_all[train_idx]
    mu_va, sigma_va, xi_va = mu_all[holdout_idx], sigma_all[holdout_idx], xi_all[holdout_idx]

    n_tr, n_ho = X_tr.shape[0], X_va.shape[0]
    L(f"all sites={n_all}, train={n_tr}, holdout={n_ho}, n_t={n_t}, K={K}")

    pd.DataFrame({"holdout_site_id": holdout_idx, "lon": sta_va[:, 0], "lat": sta_va[:, 1]}
                 ).to_csv(os.path.join(RESULT_DIR, "holdout_site_index.csv"), index=False)
    pd.DataFrame({"train_site_id": train_idx, "lon": sta_tr[:, 0], "lat": sta_tr[:, 1]}
                 ).to_csv(os.path.join(RESULT_DIR, "train_site_index.csv"), index=False)

    L("=== GEV-PIT -> standard normal (train sites only) ===")
    Z_tr = gev_pit_to_normal_with_par(X_tr, mu_tr, sigma_tr, xi_tr)
    Z_T = torch.tensor(Z_tr.T.copy(), dtype=torch.float32, device=DEVICE)
    W_tr_t = torch.tensor(W_tr, dtype=torch.float32, device=DEVICE)
    W_va_t = torch.tensor(W_va, dtype=torch.float32, device=DEVICE)

    # distance-stratified site pairs for the chi(u) training loss
    rng = np.random.default_rng(GLOBAL_SEED)
    D = squareform(pdist(sta_tr))
    iu = np.triu_indices(n_tr, k=1)
    pair_dists = D[iu]
    iu_i, iu_j = np.array(iu[0]), np.array(iu[1])
    d_max = float(pair_dists.max())
    bin_edges = np.array([0.00, 0.04, 0.08, 0.13, 0.20, 0.30, 0.45, 0.65, 1.01]) * d_max
    n_per_bin = max(100, CHI_N_PAIRS // (len(bin_edges) - 1))
    pi_parts, pj_parts = [], []
    for kbin in range(len(bin_edges) - 1):
        mask = (pair_dists >= bin_edges[kbin]) & (pair_dists < bin_edges[kbin + 1])
        n_in_bin = int(mask.sum())
        if n_in_bin == 0:
            continue
        chosen = rng.choice(np.where(mask)[0], min(n_per_bin, n_in_bin), replace=False)
        pi_parts.append(iu_i[chosen])
        pj_parts.append(iu_j[chosen])
    pi, pj = np.concatenate(pi_parts), np.concatenate(pj_parts)
    pair_i_t = torch.tensor(pi, dtype=torch.long, device=DEVICE)
    pair_j_t = torch.tensor(pj, dtype=torch.long, device=DEVICE)
    L(f"chi(u) training pairs: {len(pi)} (distance-bin balanced)")

    BS = min(BATCH_SIZE, n_t)

    # ---- Stage 1: reconstruction warm-up ----
    L(f"\n=== Stage 1: {N_EPOCHS_S1} epochs (factor off) ===")
    model = CopulaVAE(n_tr, K, n_t, factor=False).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=_LR_S1, weight_decay=1e-4)
    t0 = time.time()
    for ep in range(N_EPOCHS_S1):
        model.train()
        perm = torch.randperm(n_t, device=DEVICE)
        for st in range(0, n_t, BS):
            ids = perm[st:st + BS]
            zb = Z_T[ids]
            z_out, mu, lv, _ = model(zb, W_tr_t, t_idx=ids, det=False)
            loss, parts = stage1_loss(z_out, zb, mu, lv)
            if not torch.isfinite(loss):
                continue
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if ep % _PRINT_EVERY == 0:
            L(f"  [S1] epoch {ep:5d}  rec={parts['rec']:.4f} qm={parts['qm']:.4f} kl={parts['kl']:.4f}")
    L(f"Stage 1 done: {(time.time() - t0) / 60:.1f} min")

    # ---- Stage 2: tail-emphasized fine-tuning, best-Q99 checkpoint ----
    L(f"\n=== Stage 2: {N_EPOCHS_S2} epochs (best-Q99 tracking) ===")
    for g in opt.param_groups:
        g["lr"] = _LR_S2
    best_q99_loss, best_s2_state = float("inf"), None
    t0 = time.time()
    for ep in range(N_EPOCHS_S2):
        model.train()
        ep_q99 = []
        perm = torch.randperm(n_t, device=DEVICE)
        for st in range(0, n_t, BS):
            ids = perm[st:st + BS]
            zb = Z_T[ids]
            z_out, _, _, _ = model(zb, W_tr_t, t_idx=ids, det=False)
            loss, parts = stage2_loss(z_out, zb)
            if not torch.isfinite(loss):
                continue
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_q99.append(parts["q99"])
        avg_q99 = float(np.mean(ep_q99)) if ep_q99 else float("inf")
        if avg_q99 < best_q99_loss and ep > 100:
            best_q99_loss, best_s2_state = avg_q99, copy.deepcopy(model.state_dict())
        if ep % _PRINT_EVERY == 0:
            L(f"  [S2] epoch {ep:5d}  q99={parts['q99']:.4f} best={best_q99_loss:.5f}")
    if best_s2_state is not None:
        model.load_state_dict(best_s2_state)
        L(f"  restored best Stage-2 checkpoint, Q99 loss={best_q99_loss:.5f}")
    L(f"Stage 2 done: {(time.time() - t0) / 60:.1f} min")

    pretrained_state = copy.deepcopy(model.state_dict())
    del model

    # ---- Stage 3: factor-copula training on training sites ----
    L("\n=== Stage 3: factor-copula training ===")
    model_efc = CopulaVAE(n_tr, K, n_t, factor=True).to(DEVICE)
    h_state = model_efc.state_dict()
    for k, v in pretrained_state.items():
        if k in h_state and h_state[k].shape == v.shape:
            h_state[k] = v.clone()
        elif k in ("fc_mu.weight", "fc_lv.weight"):
            h_state[k][:K, :] = v
        elif k in ("fc_mu.bias", "fc_lv.bias"):
            h_state[k][:K] = v
    model_efc.load_state_dict(h_state)

    trainable = ("log_alpha_basis", "log_nu_basis", "log_theta_basis",
                 "logit_alpha_ps", "logit_gate_basis",
                 "fc_mu.weight", "fc_mu.bias", "fc_lv.weight", "fc_lv.bias")
    for name, p in model_efc.named_parameters():
        p.requires_grad_(name in trainable)

    groups = {"enc": [], "nu": [], "theta": [], "aps": [], "gate": []}
    for name, p in model_efc.named_parameters():
        if not p.requires_grad:
            continue
        if name == "log_nu_basis":
            groups["nu"].append(p)
        elif name == "log_theta_basis":
            groups["theta"].append(p)
        elif name == "logit_alpha_ps":
            groups["aps"].append(p)
        elif name == "logit_gate_basis":
            groups["gate"].append(p)
        else:
            groups["enc"].append(p)

    opt_efc = optim.AdamW([
        {"params": groups["enc"], "lr": _LR_S3, "weight_decay": 1e-5},
        {"params": groups["nu"], "lr": _LR_FACTOR_NU, "weight_decay": 0},
        {"params": groups["theta"], "lr": _LR_FACTOR_THETA, "weight_decay": 0},
        {"params": groups["aps"], "lr": _LR_FACTOR_APS, "weight_decay": 0},
        {"params": groups["gate"], "lr": _LR_FACTOR_GATE, "weight_decay": 0},
    ])

    model_efc.eval()
    with torch.no_grad():
        chi_target = compute_chi_soft(Z_T, pair_i_t, pair_j_t)
        gate_target = estimate_gate_target(chi_target).to(DEVICE)
    L(f"empirical training chi(top u) = {chi_target[-1].mean().item():.3f}, "
      f"gate target = {gate_target.item():.3f}")

    @torch.no_grad()
    def quick_train_eval(m):
        m.eval()
        z_out, _, _, _ = m(Z_T, W_tr_t, t_idx=None, det=True)
        if m.b.shape == (n_tr, n_t):
            z_out = z_out + m.b.T
        z_out = z_out.clamp(min=-Z_HAT_CAP, max=Z_HAT_CAP)
        X_pred = inverse_pit_gev(z_out.cpu().numpy().T, mu_tr, sigma_tr, xi_tr)
        q99_t = np.quantile(X_tr, 0.99, axis=1)
        q99_p = np.quantile(X_pred, 0.99, axis=1)
        return float(np.sqrt(np.mean((q99_t - q99_p) ** 2)))

    best_q99_val, best_state, best_ep = float("inf"), None, 0

    def run_stage(n_ep, ep_offset, tag):
        nonlocal best_q99_val, best_state, best_ep
        for ep in range(n_ep):
            if tag == "S3b" and ep == n_ep // 2:
                for g in opt_efc.param_groups:
                    g["lr"] *= _LR_DECAY_FACTOR
            model_efc.train()
            ep_chi = []
            perm = torch.randperm(n_t, device=DEVICE)
            for st in range(0, n_t, BS):
                ids = perm[st:st + BS]
                zb = Z_T[ids]
                z_recon, _, _, _ = model_efc(zb, W_tr_t, t_idx=ids, det=False)
                n_gen = max(BS * 4, 128)
                z_prior = torch.randn(n_gen, model_efc.latent_dim, device=DEVICE)
                z_gen = model_efc.decode(z_prior, W_tr_t, t_idx=None)
                loss, parts = stage3_loss(
                    z_recon, zb, z_gen, chi_target, model_efc.alpha_basis(),
                    model_efc.log_nu_basis, model_efc.log_theta_basis,
                    model_efc.gate_basis_val(), model_efc.gate_per_site(W_tr_t),
                    gate_target, pair_i_t, pair_j_t)
                if not torch.isfinite(loss):
                    continue
                opt_efc.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model_efc.parameters() if p.requires_grad], 1.0)
                opt_efc.step()
                ep_chi.append(parts["chi"])

            ep_global = ep_offset + ep
            if ep % _EVAL_EVERY == 0 and ep > _BEST_TRACK_EP_MIN:
                q = quick_train_eval(model_efc)
                if q < best_q99_val:
                    best_q99_val, best_state, best_ep = q, copy.deepcopy(model_efc.state_dict()), ep_global
            if ep % 50 == 0:
                avg_chi = float(np.mean(ep_chi)) if ep_chi else float("nan")
                a = model_efc.alpha_per_site(W_tr_t).detach().cpu().numpy()
                g = model_efc.gate_per_site(W_tr_t).detach().cpu().numpy()
                L(f"  [{tag}] epoch {ep:5d}  chi={avg_chi:.5f}  alpha={a.mean():.2f}  "
                  f"gate={g.mean():.3f}  best_train_Q99={best_q99_val:.4f}@{best_ep}")

    L(f"  Stage 3a: {N_EPOCHS_S3A} epochs")
    t0 = time.time()
    run_stage(N_EPOCHS_S3A, 0, "S3a")
    L(f"  Stage 3a done: {(time.time() - t0) / 60:.1f} min")

    L(f"  Stage 3b: {N_EPOCHS_S3B} epochs")
    for g in opt_efc.param_groups:
        g["lr"] *= _LR_DECAY_FACTOR
    t0 = time.time()
    run_stage(N_EPOCHS_S3B, N_EPOCHS_S3A, "S3b")
    L(f"  Stage 3b done: {(time.time() - t0) / 60:.1f} min")

    if best_state is not None:
        model_efc.load_state_dict(best_state)
        L(f"restored best training checkpoint @epoch {best_ep}: train Q99 RMSE={best_q99_val:.4f}")

    # ---- Holdout emulation: oracle marginal + IDW residual transfer ----
    L(f"\n=== Holdout emulation ({N_EMUL} replicates) ===")
    model_efc.eval()
    with torch.no_grad():
        alpha_train = model_efc.alpha_per_site(W_tr_t).cpu().numpy()
        nu_train = model_efc.nu_per_site(W_tr_t).cpu().numpy()
        theta_train = model_efc.theta_per_site(W_tr_t).cpu().numpy()
        gate_train = model_efc.gate_per_site(W_tr_t).cpu().numpy()
        alpha_ps_final = model_efc.alpha_ps().item()

        alpha_hold = model_efc.alpha_per_site(W_va_t).cpu().numpy()
        nu_hold = model_efc.nu_per_site(W_va_t).cpu().numpy()
        theta_hold = model_efc.theta_per_site(W_va_t).cpu().numpy()
        gate_hold = model_efc.gate_per_site(W_va_t).cpu().numpy()

        b_train = model_efc.b.detach().cpu().numpy().astype(np.float32)
        b_hold = idw_interpolate_matrix(b_train, sta_va, sta_tr, n_neighbors=8)  # Eq. kmrt-idw

        L(f"train  alpha(s): mean={alpha_train.mean():.3f} sd={alpha_train.std():.3f}")
        L(f"train  nu(s):    mean={nu_train.mean():.3f} sd={nu_train.std():.3f}")
        L(f"train  gate:     mean={gate_train.mean():.3f} sd={gate_train.std():.3f}")
        L(f"alpha_PS = {alpha_ps_final:.3f}")
        L(f"holdout alpha(s): mean={alpha_hold.mean():.3f} sd={alpha_hold.std():.3f}")
        L(f"holdout nu(s):    mean={nu_hold.mean():.3f} sd={nu_hold.std():.3f}")

        mu_enc, lv_enc = model_efc.encode(Z_T)
        emul_no_res = np.zeros((N_EMUL, n_ho, n_t), dtype=np.float32)
        emul_with_res = np.zeros((N_EMUL, n_ho, n_t), dtype=np.float32)

        for r in range(N_EMUL):
            z_samp = model_efc.reparam(mu_enc, lv_enc, det=False)
            z_hold = model_efc.decode(z_samp, W_va_t, t_idx=None).clamp(min=-Z_HAT_CAP, max=Z_HAT_CAP)
            Z_hold_no_res = z_hold.detach().cpu().numpy().T.astype(np.float32)
            Z_hold_with_res = np.clip(Z_hold_no_res + b_hold, -Z_HAT_CAP, Z_HAT_CAP).astype(np.float32)

            emul_no_res[r] = inverse_pit_gev(Z_hold_no_res, mu_va, sigma_va, xi_va)
            emul_with_res[r] = inverse_pit_gev(Z_hold_with_res, mu_va, sigma_va, xi_va)
            if (r + 1) % 20 == 0:
                L(f"  holdout emulation {r + 1}/{N_EMUL}")

        X_hat_no_res = np.median(emul_no_res, axis=0).astype(np.float32)
        X_hat_with_res = np.median(emul_with_res, axis=0).astype(np.float32)  # main result

    # per-emulation holdout fields, used for the chi(u)/ARE envelopes
    emul_dir = os.path.join(RESULT_DIR, "holdout_emulations")
    os.makedirs(emul_dir, exist_ok=True)
    for r in range(N_EMUL):
        pd.DataFrame(emul_with_res[r]).to_csv(
            os.path.join(emul_dir, f"holdout_emul_{r:03d}.csv"), index=False)
    pd.DataFrame({"holdout_site_id": holdout_idx, "lon": sta_va[:, 0], "lat": sta_va[:, 1]}
                 ).to_csv(os.path.join(RESULT_DIR, "holdout_coords.csv"), index=False)

    # training-site diagnostic emulation (direct learned residual, no IDW)
    with torch.no_grad():
        mu_enc, lv_enc = model_efc.encode(Z_T)
        emul_tr = np.zeros((N_EMUL, n_tr, n_t), dtype=np.float32)
        for r in range(N_EMUL):
            z_samp = model_efc.reparam(mu_enc, lv_enc, det=False)
            z_tr = model_efc.decode(z_samp, W_tr_t, t_idx=None) + model_efc.b.T
            z_tr = z_tr.clamp(min=-Z_HAT_CAP, max=Z_HAT_CAP)
            emul_tr[r] = inverse_pit_gev(z_tr.detach().cpu().numpy().T, mu_tr, sigma_tr, xi_tr)
        X_hat_train = np.median(emul_tr, axis=0).astype(np.float32)

    # ---- Diagnostics ----
    met_no_res = quantile_metrics(X_va, X_hat_no_res)
    met_with_res = quantile_metrics(X_va, X_hat_with_res)
    met_train = quantile_metrics(X_tr, X_hat_train)

    L("\n--- Holdout diagnostics (bias / Var / RMSE, Table redsea-main) ---")
    L(f"[with_residual] Q95: bias={met_with_res['bias_q95']:+.4f} "
      f"Var={met_with_res['var_q95']:.4f} RMSE={met_with_res['rmse_q95']:.4f} | "
      f"Q99: bias={met_with_res['bias_q99']:+.4f} Var={met_with_res['var_q99']:.4f} "
      f"RMSE={met_with_res['rmse_q99']:.4f} | field RMSE={met_with_res['field_rmse']:.4f}")
    L(f"[no_residual]   Q95: bias={met_no_res['bias_q95']:+.4f} "
      f"Var={met_no_res['var_q95']:.4f} RMSE={met_no_res['rmse_q95']:.4f} | "
      f"Q99: bias={met_no_res['bias_q99']:+.4f} Var={met_no_res['var_q99']:.4f} "
      f"RMSE={met_no_res['rmse_q99']:.4f} | field RMSE={met_no_res['field_rmse']:.4f}")
    L(f"[train diagnostic] Q95 RMSE={met_train['rmse_q95']:.4f}, Q99 RMSE={met_train['rmse_q99']:.4f}")

    # ---- Save ----
    pd.DataFrame(X_hat_with_res).to_csv(
        os.path.join(RESULT_DIR, "holdout_prediction_with_residual.csv"), index=False)
    pd.DataFrame(X_hat_no_res).to_csv(
        os.path.join(RESULT_DIR, "holdout_prediction_no_residual.csv"), index=False)
    pd.DataFrame(X_hat_train).to_csv(
        os.path.join(RESULT_DIR, "train_prediction.csv"), index=False)
    pd.DataFrame(b_hold).to_csv(
        os.path.join(RESULT_DIR, "residual_holdout_idw.csv"), index=False)

    pd.DataFrame({"site_id": train_idx, "lon": sta_tr[:, 0], "lat": sta_tr[:, 1],
                  "alpha": alpha_train, "nu": nu_train, "theta": theta_train, "gate": gate_train}
                 ).to_csv(os.path.join(RESULT_DIR, "learned_params_train.csv"), index=False)

    pd.DataFrame({
        "site_id": holdout_idx, "lon": sta_va[:, 0], "lat": sta_va[:, 1],
        "alpha": alpha_hold, "nu": nu_hold, "theta": theta_hold, "gate": gate_hold,
        "gev_loc": mu_va, "gev_scale": sigma_va, "gev_shape": xi_va,
        "observed_q95": np.nanquantile(X_va, 0.95, axis=1),
        "observed_q99": np.nanquantile(X_va, 0.99, axis=1),
        "pred_q95_with_residual": np.nanquantile(X_hat_with_res, 0.95, axis=1),
        "pred_q99_with_residual": np.nanquantile(X_hat_with_res, 0.99, axis=1),
        "pred_q95_no_residual": np.nanquantile(X_hat_no_res, 0.95, axis=1),
        "pred_q99_no_residual": np.nanquantile(X_hat_no_res, 0.99, axis=1),
    }).to_csv(os.path.join(RESULT_DIR, "holdout_quantiles.csv"), index=False)

    L(f"holdout alpha(s) mean(sd) = {alpha_hold.mean():.3f} ({alpha_hold.std():.3f})")
    L(f"holdout nu(s)    mean(sd) = {nu_hold.mean():.2f} ({nu_hold.std():.2f})")
    pd.DataFrame([{
        "dataset": "redsea_holdout",
        "alpha_mean": alpha_hold.mean(), "alpha_sd": alpha_hold.std(),
        "nu_mean": nu_hold.mean(), "nu_sd": nu_hold.std(),
        "gate_mean": gate_hold.mean(), "gate_sd": gate_hold.std(),
        "alpha_ps": alpha_ps_final,
    }]).to_csv(os.path.join(RESULT_DIR, "learned_params_summary.csv"), index=False)

    summary_rows = []
    for setting, met in [("with_residual", met_with_res),
                          ("no_residual", met_no_res),
                          ("train_diagnostic", met_train)]:
        summary_rows.append({
            "setting": setting, "n_train": n_tr, "n_holdout": n_ho, "n_t": n_t,
            "n_emul": N_EMUL, "holdout_seed": args.holdout_seed,
            "best_train_q99_rmse": best_q99_val, "best_ep": best_ep,
            "alpha_ps": alpha_ps_final, **met,
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(RESULT_DIR, "holdout_summary.csv"), index=False)

    with open(os.path.join(RESULT_DIR, "training_log.txt"), "w") as f:
        f.write("\n".join(log_lines))

    L(f"\nDone. Results written to: {RESULT_DIR}")
    L(f"Per-emulation holdout fields: {emul_dir}/holdout_emul_000..{N_EMUL - 1:03d}.csv")
    L("Postprocessing (chi(u)/ARE envelopes, plots) is done by separate scripts; "
      "see the postprocessing scripts in this repository.")


if __name__ == "__main__":
    main()